# RevuLens Pilot Experiment (Colab-Ready Notebook)
## Cumulative Pipeline: Preprocessing, Baseline, Feature Extraction & LinearSVC Model Selection

Thesis Pilot comparing frozen encoders against TF-IDF SVM baseline for Genuine (0) vs Computer-Generated (1) review classification.

In [ ]:
# --- Configuration & Paths ---
# Change these paths if running in Google Colab (e.g. '/content/fake reviews dataset.csv')
SALMINEN_DATA_PATH = r'fake reviews dataset.csv'  # update to local or colab path
RANDOM_SEED = 42
RESULTS_DIR = 'results'
DATA_CACHE_DIR = 'data'

CANDIDATE_ENCODERS = [
    'distilbert-base-multilingual-cased',
    'xlm-roberta-base',
    'jcblaise/roberta-tagalog-base'
]

import os
import re
import html
import json
import random
import time
import tracemalloc
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    brier_score_loss, confusion_matrix
)

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        else:
            torch.set_num_threads(min(4, os.cpu_count() or 1))
    except ImportError:
        pass

set_seed(RANDOM_SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_CACHE_DIR, exist_ok=True)
print('Setup complete. Seed set to', RANDOM_SEED)

In [ ]:
# --- Text Normalization ---
def normalize_text(text: Any) -> str:
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)  # remove HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # remove URLs
    text = text.lower()  # lowercase
    text = re.sub(r'([!?,.:;])\1+', r'\1', text)  # normalize repeated punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # normalize whitespace
    return text

print('Normalized sample:', normalize_text('<div>Check https://example.com!!! BEST product ever!</div>'))

In [ ]:
# --- Dataset Loader (Supports Salminen and Future Schema) ---
def load_dataset(filepath: str, verbose: bool = True) -> pd.DataFrame:
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'File not found: {filepath}')
    df = pd.read_csv(filepath)
    
    text_col = 'text_' if 'text_' in df.columns else ('text' if 'text' in df.columns else 'review')
    df['clean_text'] = df[text_col].apply(normalize_text)
    
    label_map = {
        'OR': 0, 'or': 0, 'Original': 0, 'genuine': 0, 'Genuine': 0, 0: 0, 0.0: 0, '0': 0,
        'CG': 1, 'cg': 1, 'Computer-Generated': 1, 'fake': 1, 'Fake': 1, 1: 1, 1.0: 1, '1': 1
    }
    df['label'] = df['label'].map(label_map).astype(int)
    if 'language' not in df.columns:
        df['language'] = 'English'
        
    # Drop empty and duplicates
    df = df[df['clean_text'].str.strip() != '']
    df = df.drop_duplicates(subset=['clean_text'], keep='first')
    df['text'] = df['clean_text']
    
    if verbose:
        print(f'Loaded {len(df)} valid records. Labels:\n{df["label"].value_counts()}')
    return df[['text', 'label', 'language']].reset_index(drop=True)

In [ ]:
# --- Stratified 80/10/10 Split ---
def split_dataset(df: pd.DataFrame, seed: int = RANDOM_SEED, save_path: Optional[str] = None):
    if save_path and os.path.exists(save_path):
        with open(save_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    indices = np.arange(len(df))
    labels = df['label'].values
    train_idx, temp_idx = train_test_split(indices, test_size=0.20, random_state=seed, stratify=labels)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=seed, stratify=labels[temp_idx])
    splits = {'train': train_idx.tolist(), 'val': val_idx.tolist(), 'test': test_idx.tolist()}
    if save_path:
        os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(splits, f, indent=2)
    return splits

In [ ]:
# --- Calibration & Evaluation Metrics ---
class ECEEvaluator:
    @staticmethod
    def compute(y_true: np.ndarray, prob_pos: np.ndarray, n_bins: int = 10) -> float:
        bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
        ece = 0.0
        n = len(y_true)
        for i in range(n_bins):
            b_low, b_high = bin_boundaries[i], bin_boundaries[i+1]
            in_bin = (prob_pos >= b_low) & (prob_pos <= b_high if i == n_bins - 1 else prob_pos < b_high)
            count = np.sum(in_bin)
            if count > 0:
                acc_bin = np.mean(y_true[in_bin])
                conf_bin = np.mean(prob_pos[in_bin])
                ece += (count / n) * np.abs(acc_bin - conf_bin)
        return float(ece)

def compute_metrics(y_true: np.ndarray, prob_cg: np.ndarray, threshold: float = 0.50):
    y_pred = (prob_cg >= threshold).astype(int)
    confidences = np.maximum(prob_cg, 1.0 - prob_cg)
    acc = accuracy_score(y_true, y_pred)
    prec_cg = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec_cg = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_cg = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    brier = brier_score_loss(y_true, prob_cg)
    ece = ECEEvaluator.compute(y_true, prob_cg, n_bins=10)
    corr_mask = (y_pred == y_true)
    incorr_mask = (y_pred != y_true)
    return {
        'accuracy': float(acc),
        'precision_cg': float(prec_cg),
        'recall_cg': float(rec_cg),
        'f1_cg': float(f1_cg),
        'brier_score': float(brier),
        'ece': float(ece),
        'mean_conf_correct': float(np.mean(confidences[corr_mask])) if np.any(corr_mask) else 0.0,
        'mean_conf_incorrect': float(np.mean(confidences[incorr_mask])) if np.any(incorr_mask) else 0.0,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
    }

In [ ]:
# --- TF-IDF Baseline Training & Calibration ---
def train_baseline_tfidf(train_texts, train_labels, val_texts, val_labels, c_grid=None):
    if c_grid is None:
        c_grid = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
    X_tr = vec.fit_transform(train_texts)
    X_va = vec.transform(val_texts)
    
    scaler = StandardScaler(with_mean=False)
    X_tr_sc = scaler.fit_transform(X_tr)
    X_va_sc = scaler.transform(X_va)
    
    best_c, best_f1 = c_grid[0], -1.0
    for c in c_grid:
        svc = LinearSVC(C=c, random_state=RANDOM_SEED, max_iter=2000, dual=False)
        svc.fit(X_tr_sc, train_labels)
        f1 = f1_score(val_labels, svc.predict(X_va_sc), pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_c = f1, c
            
    print(f'Selected optimal C: {best_c} (Validation F1={best_f1:.4f})')
    base_svc = LinearSVC(C=best_c, random_state=RANDOM_SEED, max_iter=2000, dual=False)
    calibrated = CalibratedClassifierCV(estimator=base_svc, method='sigmoid', cv=5)
    calibrated.fit(X_tr_sc, train_labels)
    val_probas = calibrated.predict_proba(X_va_sc)[:, 1]
    metrics = compute_metrics(val_labels, val_probas)
    return calibrated, scaler, vec, metrics, val_probas

In [ ]:
# --- Frozen Encoders & Mean Pooling Feature Extraction ---
import torch
from transformers import AutoTokenizer, AutoModel

def mean_pooling(token_embeddings, attention_mask):
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

class FeatureExtractor:
    def __init__(self, model_name: str, device: Optional[str] = None, batch_size: int = 128):
        self.model_name = model_name
        self.batch_size = batch_size
        self.device = torch.device(device) if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False
            
    def extract(self, texts: List[str], max_length: int = 128) -> np.ndarray:
        all_embs = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            enc = self.tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(self.device)
            with torch.inference_mode():
                outputs = self.model(**enc)
                pooled = mean_pooling(outputs[0], enc['attention_mask'])
                all_embs.append(pooled.cpu().numpy().astype(np.float32))
        return np.vstack(all_embs) if all_embs else np.empty((0, 768), dtype=np.float32)

def get_cache_filename(model_name: str, split_name: str, max_length: int = 128) -> str:
    safe_name = model_name.replace('/', '_').replace('-', '_')
    return f'embeddings_{safe_name}_{split_name}_len{max_length}.npy'

def get_or_compute_embeddings(model_name, split_name, texts, cache_dir=DATA_CACHE_DIR, max_length=128, batch_size=128, extractor=None):
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, get_cache_filename(model_name, split_name, max_length))
    if os.path.exists(path):
        print(f'Loading cached embeddings: {path}')
        return np.load(path)
    if extractor is None:
        extractor = FeatureExtractor(model_name, batch_size=batch_size)
    embs = extractor.extract(texts, max_length=max_length)
    np.save(path, embs)
    print(f'Saved {len(texts)} embeddings -> {path}')
    return embs

In [ ]:
# --- LinearSVC Training, Calibration & Validation Evaluation ---
def train_linear_svc_pipeline(X_train, y_train, X_val, y_val, c_grid=None, seed=RANDOM_SEED):
    if c_grid is None:
        c_grid = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
    scaler = StandardScaler(with_mean=True)
    X_tr_sc = scaler.fit_transform(X_train)
    X_va_sc = scaler.transform(X_val)
    
    best_c, best_f1 = c_grid[0], -1.0
    for c in c_grid:
        svc = LinearSVC(C=c, random_state=seed, max_iter=2000, dual=False)
        svc.fit(X_tr_sc, y_train)
        f1 = f1_score(y_val, svc.predict(X_va_sc), pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_c = f1, c
            
    base_svc = LinearSVC(C=best_c, random_state=seed, max_iter=2000, dual=False)
    calibrated = CalibratedClassifierCV(estimator=base_svc, method='sigmoid', cv=5)
    calibrated.fit(X_tr_sc, y_train)
    val_probas = calibrated.predict_proba(X_va_sc)[:, 1]
    metrics = compute_metrics(y_val, val_probas)
    return calibrated, scaler, best_c, metrics, val_probas

def select_best_pipeline(results, speed_lookup=None):
    sorted_m = sorted(results.keys(), key=lambda m: results[m]['metrics']['f1_cg'], reverse=True)
    top_m = sorted_m[0]
    top_f1 = results[top_m]['metrics']['f1_cg']
    within_1pt = [m for m in sorted_m if (top_f1 - results[m]['metrics']['f1_cg']) <= 0.01]
    if len(within_1pt) == 1 or speed_lookup is None:
        return top_m, f'Highest F1={top_f1:.4f}'
    fastest = max(within_1pt, key=lambda m: speed_lookup.get(m, 0.0))
    return fastest, f'Selected {fastest} (faster/smaller within 1pt of {top_m})'

def evaluate_seed_stability(X_train, y_train, X_val, y_val, best_c, seeds=[42, 7, 123], tune_c_inner=True, c_grid=None):
    import scipy.sparse as sp
    is_sparse = sp.issparse(X_train)
    X_dev = sp.vstack([X_train, X_val]) if is_sparse else np.vstack([X_train, X_val])
    y_dev = np.concatenate([y_train, y_val])
    val_size = len(y_val)
    if c_grid is None:
        c_grid = [0.001, 0.003, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
    f1_list, brier_list, chosen_c_list = [], [], []
    for s in seeds:
        tr_sub, va_sub = train_test_split(np.arange(len(y_dev)), test_size=val_size, random_state=s, stratify=y_dev)
        chosen_c = best_c
        if tune_c_inner and len(tr_sub) >= 20:
            in_tr, in_va = train_test_split(np.arange(len(tr_sub)), test_size=0.15, random_state=s, stratify=y_dev[tr_sub])
            sc_in = StandardScaler(with_mean=not is_sparse)
            X_in_tr = sc_in.fit_transform(X_dev[tr_sub[in_tr]])
            X_in_va = sc_in.transform(X_dev[tr_sub[in_va]])
            best_in_f1 = -1.0
            for c in c_grid:
                svc_in = LinearSVC(C=c, random_state=s, max_iter=2000, dual=False)
                svc_in.fit(X_in_tr, y_dev[tr_sub[in_tr]])
                preds = svc_in.predict(X_in_va)
                score = f1_score(y_dev[tr_sub[in_va]], preds, pos_label=1, zero_division=0)
                if score > best_in_f1:
                    best_in_f1 = score
                    chosen_c = c
        chosen_c_list.append(chosen_c)
        scaler = StandardScaler(with_mean=not is_sparse)
        X_tr = scaler.fit_transform(X_dev[tr_sub])
        X_va = scaler.transform(X_dev[va_sub])
        base_svc = LinearSVC(C=chosen_c, random_state=s, max_iter=2000, dual=False)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=s)
        calibrated = CalibratedClassifierCV(estimator=base_svc, method='sigmoid', cv=cv)
        calibrated.fit(X_tr, y_dev[tr_sub])
        probs = calibrated.predict_proba(X_va)[:, 1]
        m = compute_metrics(y_dev[va_sub], probs)
        f1_list.append(m['f1_cg'])
        brier_list.append(m['brier_score'])
    return {
        'seeds': seeds,
        'chosen_c': chosen_c_list,
        'f1_scores': f1_list,
        'mean_f1': round(float(np.mean(f1_list)), 4),
        'std_f1': round(float(np.std(f1_list)), 4),
        'brier_scores': brier_list,
        'mean_brier': round(float(np.mean(brier_list)), 4),
        'std_brier': round(float(np.std(brier_list)), 4)
    }

def plot_confusion_matrices(pipeline_results, output_path='results/confusion_matrices.png'):
    n = len(pipeline_results)
    fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(10, 4 * ((n + 1) // 2)))
    axes = np.array([axes]).flatten()
    for idx, (name, res) in enumerate(pipeline_results.items()):
        cm = np.array(res['metrics']['confusion_matrix'])
        ax = axes[idx]
        ax.imshow(cm, cmap='Blues')
        ax.set_title(f"{name.split('/')[-1]}\nF1={res['metrics']['f1_cg']:.4f}")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path

## Step 10 & 13: Test Set Evaluation & Reliability Diagrams

- **Step 10**: Evaluate selected pipeline (`distilbert-base-multilingual-cased`) and baseline (`TF-IDF + LinearSVC`) **ONCE** on the unseen test set.
- **Step 13**: Plot calibration reliability diagrams for validation and test splits.

In [ ]:
# --- Test Set Evaluation & Reliability Diagram Functions ---
from sklearn.calibration import calibration_curve

def evaluate_test_set(calibrated_clf, scaler, X_test, y_test):
    X_test_scaled = scaler.transform(X_test)
    test_probas = calibrated_clf.predict_proba(X_test_scaled)[:, 1]
    return compute_metrics(y_test, test_probas), test_probas

def evaluate_baseline_test_set(calibrated_clf, scaler, vectorizer, test_texts, y_test):
    X_test_tfidf = vectorizer.transform(test_texts)
    X_test_scaled = scaler.transform(X_test_tfidf)
    test_probas = calibrated_clf.predict_proba(X_test_scaled)[:, 1]
    return compute_metrics(y_test, test_probas), test_probas

def plot_reliability_diagram(diagram_data, output_path='results/reliability_diagram.png', n_bins=10):
    n_splits = len(diagram_data)
    fig, axes = plt.subplots(1, n_splits, figsize=(6.5 * n_splits, 5))
    if n_splits == 1:
        axes = [axes]
    for idx, (split_name, data) in enumerate(diagram_data.items()):
        ax = axes[idx]
        y_true, y_prob = np.asarray(data['y_true']), np.asarray(data['y_prob'])
        ece_val = ECEEvaluator.compute(y_true, y_prob, n_bins=n_bins)
        brier_val = float(brier_score_loss(y_true, y_prob))
        prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy='uniform')
        ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
        color = '#1f77b4' if idx == 0 else '#2ca02c'
        label_txt = f"{data.get('label', split_name)}\n(ECE={ece_val:.4f}, Brier={brier_val:.4f})"
        ax.plot(prob_pred, prob_true, marker='s', linewidth=2, color=color, label=label_txt)
        ax.set_title(f'Reliability Diagram: {split_name} Set', fontsize=11, fontweight='bold')
        ax.set_xlabel('Mean Predicted Probability P(CG)', fontsize=10)
        ax.set_ylabel('Fraction of Positives (Empirical CG Rate)', fontsize=10)
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    plt.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close()
    return output_path

In [ ]:
# --- Execute Test Set Evaluation (Day 4) ---
test_results = run_day_4()
print('Test Set Evaluation and Reliability Diagrams Complete!')

## Step 12: FiReCS Out-of-Domain False Alarm Check

- **Step 12**: Evaluate fitted classifiers on unlabeled Filipino/Taglish reviews (FiReCS).
- **Mandatory Caveat**: FiReCS is strictly presumed genuine and encoders were trained only on English. A high CG (false alarm) rate is expected.

In [ ]:
# --- FiReCS Out-of-Domain Check Functions ---
def load_firecs_dataset(train_path=FIRECS_TRAIN_PATH, test_path=FIRECS_TEST_PATH, verbose=True):
    dfs = []
    for p, sp in [(train_path, 'train'), (test_path, 'test')]:
        if os.path.exists(p):
            d = pd.read_csv(p)
            col = 'review' if 'review' in d.columns else 'text'
            d = d[[col]].copy().rename(columns={col: 'raw_text'})
            d['source_split'] = sp
            dfs.append(d)
    df = pd.concat(dfs, ignore_index=True)
    df['clean_text'] = df['raw_text'].apply(normalize_text)
    df = df[df['clean_text'].str.strip() != ''].drop_duplicates(subset=['clean_text']).reset_index(drop=True)
    df['language'] = 'Filipino/Taglish'
    df['word_count'] = df['clean_text'].str.split().str.len()
    return df

def compute_confidence_distribution(probas_cg, threshold=0.50):
    probas_cg = np.asarray(probas_cg, dtype=float)
    confidences = np.maximum(probas_cg, 1.0 - probas_cg)
    preds_cg = (probas_cg >= threshold).astype(int)
    total = len(probas_cg)
    cg_cnt = int(np.sum(preds_cg == 1))
    return {
        'total_reviews': total,
        'predicted_cg_count': cg_cnt,
        'false_alarm_rate_percent': round(float(cg_cnt / total * 100.0), 2),
        'mean_confidence': round(float(np.mean(confidences)), 4),
        'median_confidence': round(float(np.median(confidences)), 4),
        'confidence_buckets': {
            'high_gte_80': int(np.sum(confidences >= 0.80)),
            'moderate_60_to_80': int(np.sum((confidences >= 0.60) & (confidences < 0.80))),
            'low_50_to_60': int(np.sum((confidences >= 0.50) & (confidences < 0.60)))
        }
    }

def evaluate_firecs_pipeline(calibrated_clf, scaler, X_features):
    X_scaled = scaler.transform(X_features)
    probas = calibrated_clf.predict_proba(X_scaled)[:, 1]
    confs = np.maximum(probas, 1.0 - probas)
    return compute_confidence_distribution(probas), probas, confs


In [ ]:
# --- Execute FiReCS False Alarm Check (Day 5) ---
firecs_results = run_day_5()
print('FiReCS Check Complete! Results saved to results/firecs_check.csv and results/firecs_summary.md')

## Step 14: Single-Review Speed Check & SHAP Latency Benchmark

- **Step 14**: Measure single-review latency (tokenization + embedding + classification) per encoder for browser extension suitability.
- **SHAP Check**: Measure explanation time for one review with background sample limits.

In [ ]:
# --- Single-Review Speed & SHAP Benchmark Functions ---
def benchmark_single_review_latency(pipeline_type, pipeline_obj, test_review='Great product works well', n_warmup=3, n_runs=20):
    import time
    clean = normalize_text(test_review)
    for _ in range(n_warmup):
        if pipeline_type == 'tfidf':
            x = pipeline_obj['scaler'].transform(pipeline_obj['vectorizer'].transform([clean]))
            _ = pipeline_obj['clf'].predict_proba(x)
        else:
            emb = pipeline_obj['extractor'].extract([clean], max_length=pipeline_obj.get('max_length', 128))
            x = pipeline_obj['scaler'].transform(emb)
            _ = pipeline_obj['clf'].predict_proba(x)
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        if pipeline_type == 'tfidf':
            x = pipeline_obj['scaler'].transform(pipeline_obj['vectorizer'].transform([clean]))
            _ = pipeline_obj['clf'].predict_proba(x)[0, 1]
        else:
            emb = pipeline_obj['extractor'].extract([clean], max_length=pipeline_obj.get('max_length', 128))
            x = pipeline_obj['scaler'].transform(emb)
            _ = pipeline_obj['clf'].predict_proba(x)[0, 1]
        times.append((time.perf_counter() - t0) * 1000.0)
    mean_ms = float(np.mean(times))
    return {'mean_ms': round(mean_ms, 2), 'p95_ms': round(float(np.percentile(times, 95)), 2), 'seconds_per_review': round(mean_ms/1000.0, 5), 'reviews_per_sec': round(1000.0/mean_ms, 1)}

def benchmark_shap_latency(model_obj, background_data, test_instance, max_samples=100, explainer_mode='linear'):
    import time
    import shap
    bg = background_data[:max_samples]
    t0 = time.perf_counter()
    if hasattr(model_obj, 'calibrated_classifiers_'):
        base_clf = model_obj.calibrated_classifiers_[0].estimator
    else:
        base_clf = model_obj
    explainer = shap.LinearExplainer(base_clf, bg)
    _ = explainer(test_instance.reshape(1, -1))
    elapsed = time.perf_counter() - t0
    return {'shap_version': shap.__version__, 'background_sample_limit': len(bg), 'seconds_per_review': round(elapsed, 4), 'latency_ms': round(elapsed * 1000.0, 2)}


In [ ]:
# --- Execute Speed & SHAP Benchmark (Day 6) ---
speed_results = run_day_6()
print('Speed & SHAP Benchmarks Complete! Results saved to results/speed_benchmark.json and results/speed_benchmark.md')

## Step 15: Final Outputs & Consolidated Report Generation

- Generate `results/results_table.csv` and `results/results_table.md`.
- Generate `results/REPORT.md` (at most one page, plain language, with exact package versions and required caveats).

In [ ]:
# --- Consolidated Results & Report Generation ---
def run_final_report():
    return run_day_7()


In [ ]:
# --- Execute Final Report Generation (Day 7) ---
final_outputs = run_day_7()
print('RevuLens Pilot Final Deliverables Generated Successfully!')